# PaisajIA en Google Colab (GPU T4)

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

Perfil T4 que aplica este notebook:

| Ajuste | Valor | Motivo |
|---|---|---|
| `PAISAJIA_DTYPE` | `fp16` | la T4 (capacidad 7.5) no acelera bf16 |
| `PAISAJIA_GEN_SIZE` | `768` | ~1.8× más rápido que 1024 px; la salida se normaliza a 1024×1024 |
| `PAISAJIA_TE_QUANT` | `4bit` | codificador de texto de FLUX (Qwen3 4B) en 4 bits: FLUX entero cabe en los 15 GB de la T4 (~11-12 GB) |
| `PAISAJIA_OLLAMA_NUM_GPU` | `0` | si usas Qwen local, corre en CPU y deja la GPU a FLUX |

Sin la cuantización, FLUX necesitaría descarga a CPU (~16 GB de RAM), y Colab gratis solo da ~12.7 GB.

Pasos: 1 · GPU → 2 · código → 3 · instalación → 4 · configuración → 5 · modelos → 6 · prueba rápida → 7 · app → 8 · evaluación (opcional) → 9 · guardar resultados.

## 1 · Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h | head -2
!df -h /content | tail -1

## 2 · Obtener el código

En tu PC (PowerShell, desde la carpeta del proyecto) ejecuta `.\scripts\colab\empaquetar.ps1`. Eso crea `paisajia_colab.zip`
**sin** credenciales (`.env`, `Mitsuo/`), `.venv` ni resultados. Luego elige una opción:

- `"drive"`: sube el .zip a *Mi unidad* de Google Drive (recomendado: lo reutilizas en cada sesión);
- `"subir"`: lo subes ahora desde el navegador;
- `"git"` (por defecto): clona el repositorio del proyecto en GitHub. Si es privado, usa `drive` o `subir`.

In [ ]:
FUENTE = "git"                                     # git | drive | subir
ZIP_DRIVE = "/content/drive/MyDrive/paisajia_colab.zip"
REPO_URL = "https://github.com/MitsuoMurakami/Proyecto2_Planificacion.git"
PROYECTO = "/content/paisajia"

import os, shutil, zipfile
if FUENTE in ("drive",):
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = ZIP_DRIVE
elif FUENTE == "subir":
    from google.colab import files
    zip_path = "/content/" + next(iter(files.upload()))
if FUENTE == "git":
    !git clone -q {REPO_URL} {PROYECTO}
else:
    shutil.rmtree(PROYECTO, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(PROYECTO)
%cd {PROYECTO}
!find scripts -name "*.sh" -exec sed -i 's/$//' {} +
!ls

## 3 · Instalar dependencias (~2-4 min)

Se conserva el `torch` que ya trae Colab. Pon `USAR_OLLAMA = True` solo si quieres el agente Qwen 2.5 3B local (corre en CPU; ~1 min por resumen).

In [ ]:
USAR_OLLAMA = False
OPCION = "--ollama" if USAR_OLLAMA else ""
!bash scripts/colab/setup.sh {OPCION}

## 4 · Configuración

Credenciales en los **Secretos** de Colab (icono de llave 🔑, a la izquierda), nunca escritas en el notebook:

- `HF_TOKEN`: necesario si el modelo de Hugging Face pide aceptar la licencia;
- `ANTHROPIC_API_KEY`: opcional, para el agente con Claude;
- `PAISAJIA_APP_PASSWORD`: **recomendado**. El enlace `--share` de Gradio es público, y con esta clave la app pide usuario `paisajia` y esa contraseña.

`GUARDAR_EN_DRIVE = True` guarda el registro (SQLite + JSON) en Drive para que no se pierda al cerrar la sesión.

In [ ]:
LLM = "auto"            # auto (Claude si hay clave, si no simulado) | claude | ollama | simulado
GUARDAR_EN_DRIVE = False

import os
from google.colab import userdata
for k in ("HF_TOKEN", "ANTHROPIC_API_KEY", "PAISAJIA_APP_PASSWORD"):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass

os.environ.update({
    "PAISAJIA_DTYPE": "fp16",
    "PAISAJIA_GEN_SIZE": "768",
    "PAISAJIA_TE_QUANT": "4bit",
    "PAISAJIA_OLLAMA_NUM_GPU": "0",
    "PAISAJIA_LLM": LLM,
    "GRADIO_ANALYTICS_ENABLED": "False",
    "PYTHONIOENCODING": "utf-8",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["PAISAJIA_DATA_DIR"] = "/content/drive/MyDrive/paisajia_data"
if LLM == "auto":
    LLM = "claude" if os.getenv("ANTHROPIC_API_KEY") else "simulado"
print("Agente:", LLM, "· HF_TOKEN:", "sí" if os.getenv("HF_TOKEN") else "no",
      "· contraseña de la app:", "sí" if os.getenv("PAISAJIA_APP_PASSWORD") else "NO (enlace público sin protección)")

## 5 · Descargar modelos (~10 min)

FLUX.2 [klein] 4B + DINOv2/CLIP/LPIPS. Añade `sd` para la línea base (SD 1.5 + ControlNet Depth, ~6 GB más). Los modelos quedan en el disco temporal de Colab, así que hay que volver a descargarlos en cada sesión.

In [ ]:
!python scripts/download_models.py --only flux vision
# !python scripts/download_models.py --only sd

## 6 · Prueba rápida

Genera una ronda (5 imágenes) con FLUX y un resumen del agente. Reporta segundos por imagen, pico de VRAM y si el resumen pasó el verificador. La primera imagen tarda más (calentamiento).

In [ ]:
!pytest -q
!python scripts/smoke_test.py --generator flux --llm {LLM} --out results/smoke-colab

In [ ]:
from IPython.display import Image as IPImage, display
import glob
for f in sorted(glob.glob("results/smoke-colab/*.jpg")):
    print(f); display(IPImage(f, width=384))

## 7 · Lanzar la app

Abre el enlace `https://….gradio.live` que aparece abajo. La celda sigue ejecutándose mientras la app está activa; detenla con ■ para terminar.
Para la línea base usa `--generator sd`.

In [ ]:
!python app.py --generator flux --llm {LLM} --share

## 8 · Evaluación de generación (opcional)

Necesita `data/lhq_subset` (empaqueta con `.\scripts\colab\empaquetar.ps1 -ConDatos`, o pon la carpeta en Drive y cambia `IMAGENES`).
En una T4, cada imagen × 5 ejes con FLUX tarda ~1-2 min. Usa `--limit` para que la sesión de Colab no expire a mitad de la evaluación.

In [ ]:
IMAGENES = "data/lhq_subset"
!python scripts/download_models.py --only sd
!python scripts/eval_generation.py --methods flux sd --images {IMAGENES} --limit 10 --out results/generation-colab

## 9 · Guardar resultados en Drive

El disco de Colab se borra al cerrar la sesión.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/paisajia_results
!cp -r results/* /content/drive/MyDrive/paisajia_results/
!ls /content/drive/MyDrive/paisajia_results